In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

# ── Configuration ──────────────────────────────────────────────────────────────
LIST_URL = (
    "https://www.kultunaut.dk/perl/arrlist/type-nynaut"
    "?Area=Kolding-storkommune"
    "&ArrSlutdato=5%2F6%202026"
    "&ArrStartdato=1%2F1%202026"
    "&Order=ArrStartdato"
    "&nearmeradius=2000"
    "&periode="
)
PAGINATE_URL = (
    "https://www.kultunaut.dk/perl/arrlist2/type-nynaut"
    "?startnr={start}"
    "&Area=Kolding-storkommune"
    "&ArrSlutdato=5%2F6%202026"
    "&ArrStartdato=1%2F1%202026"
    "&Order=ArrStartdato"
    "&nearmeradius=2000"
    "&periode="
)
PAGE_SIZE = 13
DELAY = 0.5  # Throttling time between hits to be safe

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "da-DK,da;q=0.9,en;q=0.8",
}

def parse_page_events(soup):
    page_events = []
    
    # Target the main product anchor tags you found via Inspect element
    event_containers = soup.find_all("a", class_="product-content")
    
    # Fallback to structural regex matching if some pages render a legacy layout list view
    if not event_containers:
        event_containers = soup.find_all("a", href=re.compile(r"/perl/arrmore/type-nynaut"))

    for container in event_containers:
        # 1. Title Extraction
        title_tag = container.find("h3")
        title = title_tag.get_text(strip=True) if title_tag else "Untitled Event"
        
        # 2. Description Extraction
        desc_tag = container.find("div", class_="arr-description")
        description = desc_tag.get_text(strip=True) if desc_tag else "No description available"
        
        # 3. Date & Time Extraction from the <time> tag inside .kult-month-day
        time_tag = container.find("time")
        
        date_clean = "Unknown Date"
        time_clean = "See description"
        
        if time_tag:
            raw_time_text = time_tag.get_text(strip=True)
            
            # If a venue name b-tag leaks into the time string, drop it to isolate temporal data
            bold_tag = container.find("b")
            if bold_tag:
                venue_text = bold_tag.get_text(strip=True)
                raw_time_text = raw_time_text.replace(venue_text, "").strip().rstrip(",")

            # Extract time configurations (e.g., matching shapes like "10 a.m.", "14:30")
            time_match = re.search(r"\b(\d{1,2}(?:\s*[a-p]\.m\.)|(?:\d{2}[:.]\d{2}))", raw_time_text, re.IGNORECASE)
            if time_match:
                time_clean = time_match.group(1)
                
            # Strip out time configurations from the raw text to leave just the clean date string
            date_clean = re.sub(r"\b\d{1,2}\s*[a-p]\.m\..*$", "", raw_time_text, flags=re.IGNORECASE).strip()
            # General fallback check to clear out raw Danish 'kl' time strings if present
            date_clean = re.sub(r"kl\..*$", "", date_clean, flags=re.IGNORECASE).strip().rstrip(",").strip()

        page_events.append({
            "Title": title,
            "Description": description,
            "Date": date_clean,
            "Time": time_clean
        })
        
    return page_events

# ── Main Scrape Execution Loop ─────────────────────────────────────────────────
if __name__ == "__main__":
    all_events = []

    print("[+] Loading primary landing index...")
    response = requests.get(LIST_URL, headers=HEADERS)
    soup = BeautifulSoup(response.text, "html.parser")

    # Locate the total event counter to set up pagination boundaries
    total_match = re.search(r"Viser\s+(\d+)\s+events", soup.get_text())
    total_events = int(total_match.group(1)) if total_match else 150
    print(f"[i] Discovered {total_events} available regional listings.")

    # Harvest first page details
    first_page_results = parse_page_events(soup)
    all_events.extend(first_page_results)
    print(f"    → Registered {len(first_page_results)} entries from page 1.")

    # Loop over the remaining items via pagination
    start_idx = PAGE_SIZE + 1
    page_num = 2
    
    while start_idx <= total_events:
        time.sleep(DELAY)
        print(f"[+] Requesting entries starting at index position {start_idx} (Page {page_num})...")
        
        response = requests.get(PAGINATE_URL.format(start=start_idx), headers=HEADERS)
        soup = BeautifulSoup(response.text, "html.parser")
        
        page_results = parse_page_events(soup)
        if not page_results:
            print("    [i] No more items found on this layout format. Ending pagination sequence.")
            break
            
        all_events.extend(page_results)
        print(f"    → Registered {len(page_results)} items.")
        
        start_idx += PAGE_SIZE
        page_num += 1

    # ── Save Clean Dataset ─────────────────────────────────────────────────────
    df = pd.DataFrame(all_events).drop_duplicates(subset=["Title", "Date"])
    output_filename = "clean_kolding_events.csv"
    df.to_csv(output_filename, index=False, encoding="utf-8-sig")

    print(f"\n✅ Pipeline Complete! Extracted {len(df)} unique events.")
    print(f"File saved cleanly to: '{output_filename}'")

Guide to culture events, music, theatre and exhibitions - Find it all on KultuNaut KultuNaut - take time to experience English Danish Swedish English German toggle menu Ukraine Literature Scout Role Playing Game Health Chess Free Storytelling Dogs Entrepreneur Evening Courses Folk dance Rootszone Electronic/Club/DJ Viking/Medieval Knitting/weaving Bicycle Tours Church music Circus Garden/plants Choir singing More categories Search Advanced search Search places Search organizers Search cinema Your shortlist Add event Edit event Send tips Login Get data Widget Special calendar Wednesday 27 May 125548 events Search Choose geography Near my position Entire Denmark Region The Capital Bornholm municipality Greater Copenhagen Copenhagen and Fredriksberg Cph. C Cph. East Cph. North Cph. West Cph. South Cph. NV Cph. SV Frederiksberg Albertslund municipality Ballerup municipality Brøndby municipality Dragør municipality Gentofte municipality Gladsaxe municipality Glostrup municipality Herlev mun

Cleaninig event data

In [14]:
#cleaning clean_kolding_events.csv

import pandas as pd
import re

events = pd.read_csv("clean_kolding_events.csv")

# Period covered by the movement dataset
period_start = pd.Timestamp("2026-02-09")
period_end = pd.Timestamp("2026-05-08 23:59:59")

danish_months = {
    "jan": 1,
    "feb": 2,
    "mar": 3,
    "apr": 4,
    "maj": 5,
    "jun": 6,
    "jul": 7,
    "aug": 8,
    "sep": 9,
    "okt": 10,
    "nov": 11,
    "dec": 12
}

def parse_single_date(text, fallback_year=2026):
    text = str(text).lower()
    
    # Remove Danish weekday abbreviations
    text = re.sub(r"\b(man|tir|ons|tor|fre|lør|søn)\.\s*", "", text)
    
    match = re.search(
        r"(\d{1,2})\.\s*([a-zæøå]{3,})\.?(?:\s+(20\d{2}))?",
        text
    )
    
    if not match:
        return pd.NaT
    
    day = int(match.group(1))
    month_text = match.group(2)[:3]
    month = danish_months.get(month_text)
    year = int(match.group(3)) if match.group(3) else fallback_year
    
    if month is None:
        return pd.NaT
    
    try:
        return pd.Timestamp(year=year, month=month, day=day)
    except:
        return pd.NaT


def parse_event_date_range(date_text):
    date_text = str(date_text)
    
    years = re.findall(r"20\d{2}", date_text)
    fallback_year = int(years[-1]) if years else 2026
    
    parts = re.split(r"\s+til\s+", date_text, maxsplit=1)
    
    if len(parts) == 2:
        end_date = parse_single_date(parts[1], fallback_year=fallback_year)
        
        if pd.notna(end_date):
            start_date = parse_single_date(parts[0], fallback_year=end_date.year)
        else:
            start_date = parse_single_date(parts[0], fallback_year=fallback_year)
    else:
        start_date = parse_single_date(date_text, fallback_year=fallback_year)
        end_date = start_date
    
    return start_date, end_date


def parse_event_time(time_text):
    if pd.isna(time_text):
        return 12, 0
    
    text = str(time_text)
    
    # Finds 10:00, 10.00, 08.45, etc.
    match = re.search(r"(\d{1,2})[:.](\d{2})", text)
    
    if match:
        return int(match.group(1)), int(match.group(2))
    
    # If time is missing, use noon as approximation
    return 12, 0

events_clean = events.copy()

# Create event category by removing the title from "Event Type"
events_clean["event_category"] = events_clean.apply(
    lambda row: str(row["Event Type"]).replace(str(row["Title"]), "").strip(),
    axis=1
)

events_clean["event_category"] = events_clean["event_category"].replace("", "Unknown")

# Parse event start and end dates
parsed_dates = events_clean["Date"].apply(parse_event_date_range)

events_clean["event_start"] = parsed_dates.apply(lambda x: x[0])
events_clean["event_end"] = parsed_dates.apply(lambda x: x[1])

# Parse time
parsed_times = events_clean["Time"].apply(parse_event_time)

events_clean["event_hour"] = parsed_times.apply(lambda x: x[0])
events_clean["event_minute"] = parsed_times.apply(lambda x: x[1])

events_clean["event_datetime"] = (
    events_clean["event_start"]
    + pd.to_timedelta(events_clean["event_hour"], unit="h")
    + pd.to_timedelta(events_clean["event_minute"], unit="m")
)

# Keep only events that overlap with the movement dataset period
events_clean = events_clean[
    (events_clean["event_start"].notna()) &
    (events_clean["event_end"].notna()) &
    (events_clean["event_start"] <= period_end) &
    (events_clean["event_end"] >= period_start)
].copy()

# Remove duplicates
events_clean = events_clean.drop_duplicates(
    subset=["event_category", "Title", "event_start", "event_end", "Time", "Event URL"]
)

# Keep only useful columns
events_clean = events_clean[
    [
        "event_category",
        "event_start",
        "event_end",
        "event_datetime",
        "Time",
        "Event URL"
    ]
]

events_clean.head()

events_clean.to_csv("clean_events_for_visualization.csv", index=False, encoding="utf-8-sig")

Visulalization heatmap

Timeline with events

In [18]:
import pandas as pd
import plotly.graph_objects as go
import numpy as np

# Load data
movement = pd.read_csv("../movement_stats_hourly.csv")
events = pd.read_csv("clean_events_for_visualization.csv")

# Define period
start_date = pd.Timestamp("2026-02-09")
end_date = pd.Timestamp("2026-05-06 23:59:59")

# Clean movement timestamps
movement["timestamp"] = pd.to_datetime(
    movement["timestamp"],
    errors="coerce",
    utc=True
)

movement["timestamp"] = (
    movement["timestamp"]
    .dt.tz_convert("Europe/Copenhagen")
    .dt.tz_localize(None)
)

# Filter movement to the correct period
movement = movement[
    (movement["timestamp"] >= start_date) &
    (movement["timestamp"] <= end_date)
].copy()

# Keep pedestrians only
pedestrians = movement[movement["category"] == "pedestrian"].copy()

hourly_people = (
    pedestrians
    .groupby("timestamp", as_index=False)["amount"]
    .sum()
)

# Clean event datetime
events["event_datetime"] = pd.to_datetime(
    events["event_datetime"],
    errors="coerce"
)

# Filter events to the same period
events = events[
    (events["event_datetime"] >= start_date) &
    (events["event_datetime"] <= end_date)
].copy()

# Plot
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=hourly_people["timestamp"],
    y=hourly_people["amount"],
    mode="lines",
    name="Pedestrian movement"
))

fig.add_trace(go.Scatter(
    x=events["event_datetime"],
    y=[hourly_people["amount"].max() * 1.05] * len(events),
    mode="markers",
    name="Events",
    text=events["event_category"],
    hovertemplate=(
        "Event type: <b>%{text}</b><br>"
        "Date/time: %{x}<extra></extra>"
    )
))

fig.update_layout(
    title="Pedestrian movement timeline with event types",
    xaxis_title="Date and time",
    yaxis_title="Pedestrian movement",
    height=600,
    hovermode="closest"
)

# Force x-axis to show only this period
fig.update_xaxes(range=[start_date, end_date])

fig.show()